# 01 — Text & Language Fundamentals

**Learning objective.** Understand corpus, document, sentence, token, vocabulary, type/token counts, OOV, and why text is structurally different from tabular data.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**raw documents → segmentation → tokens/vocabulary → measurable language structure**

The key question is not “which API do I call?” but **which representation changes next when I change a control?**

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Change the **document boundary** | document count and context size change | statistics such as DF, length and labels can change |
| Change the **token definition** | token count and vocabulary change | all vectorizers/sequence models see a different input space |
| Raise a **minimum-frequency cutoff** | rare vocabulary shrinks | memory improves but rare domain terms can disappear |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. If one long email is split into five sentences, what happens to document count and average length?
2. If `credit-card` becomes two tokens instead of one, what changes downstream?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Use these primitives whenever defining corpus, token, vocabulary and OOV behavior.

### When not to use / caution
Do not assume 'word' or 'document' is universal; the task determines the useful unit.

### Debugging lens
Unexpected model behavior often starts with an unexpected unit of analysis—inspect raw → document → sentence → token counts.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


## Core vocabulary
- **Corpus:** a collection of documents.
- **Document:** one independently meaningful text unit.
- **Token:** a unit produced by a tokenizer.
- **Type:** a unique token value.
- **Vocabulary:** set/mapping of known token types.
- **OOV:** out-of-vocabulary token at inference time.
- **Sequence length:** number of tokens in a document; important for padding, truncation, cost and latency.

In [2]:
docs = [
 'NLP turns language into representations.',
 'Language contains ambiguity and context.',
 'Representations make language computable.'
]
tokens = [re.findall(r"\b\w+\b", d.lower()) for d in docs]
vocab = sorted(set(t for row in tokens for t in row))
print('documents:', len(docs))
print('tokens:', sum(map(len,tokens)))
print('types:', len(vocab))
print('type/token ratio:', round(len(vocab)/sum(map(len,tokens)),3))
print('vocabulary:', vocab)

documents: 3
tokens: 14
types: 11
type/token ratio: 0.786
vocabulary: ['ambiguity', 'and', 'computable', 'contains', 'context', 'into', 'language', 'make', 'nlp', 'representations', 'turns']


In [3]:
known=set(vocab)
query='context drives useful embeddings'
qtoks=re.findall(r"\b\w+\b", query.lower())
print(pd.DataFrame({'token':qtoks,'known':[t in known for t in qtoks]}).to_string(index=False))

     token  known
   context   True
    drives  False
    useful  False
embeddings  False


---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Differentiate token and type
- Explain OOV and sequence length as production constraints